# sklearn in 20 minutes

This is the shared primer for the group. It covers the two sklearn pieces we
will use everywhere: **`ColumnTransformer`** and **`Pipeline`**.

You already know `cross_validate`. By the end of this notebook you should be
able to explain why this is safe:

```python
pipe = Pipeline([("prep", preprocessor), ("clf", model)])
cross_validate(pipe, X_raw, y, cv=folds)
```

and why this is not:

```python
X_scaled = preprocessor.fit_transform(X_raw)   # sees every row
cross_validate(model, X_scaled, y, cv=folds)
```

The difference is not style. It is the difference between a number you can
trust and a number that is too good to be true.


## 1. The estimator API

Every sklearn object is an *estimator*: it has `fit`, and then either
`transform` (preprocessing) or `predict`/`predict_proba` (modelling).
Configuration lives in `get_params()`; `clone()` copies that configuration
without the fitted state.

`Pipeline` and `ColumnTransformer` are estimators too, so they compose: a
pipeline can contain a column transformer, and a grid search can contain a
pipeline.


In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler

raw = pd.DataFrame({"height": [160, 170, 180, 190], "weight": [60, 70, 80, 90]})
scaler = StandardScaler()

print("fitted before fit?", hasattr(scaler, "mean_"))
scaler.fit(raw)
print("fitted after fit? ", hasattr(scaler, "mean_"))
print("clone is unfitted?", not hasattr(clone(scaler), "mean_"))
print("scaled values:")
print(scaler.transform(raw).round(2))


fitted before fit? False
fitted after fit?  True
clone is unfitted? True
scaled values:
[[-1.34 -1.34]
 [-0.45 -0.45]
 [ 0.45  0.45]
 [ 1.34  1.34]]


## 2. The leakage trap

We generate a small dataset with **200 columns but only 2 informative ones**.
A feature selector that sees all labels will pick noise columns that happen to
correlate with `y` on this sample. Cross-validation then reports a much better
score than the model deserves.

This is exactly the bug in our first feature-selection notebook: `SelectFromModel`
was fitted on all training rows *before* cross-validation.


In [3]:
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

# generates sample dataset
X_demo, y_demo = make_classification(
    n_samples=80,
    n_features=200,
    n_informative=2,
    n_redundant=0,
    flip_y=0.25,
    class_sep=0.8,
    random_state=0,
)
demo_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X_demo.shape, y_demo.shape)


(80, 200) (80,)


In [4]:
# WRONG: the selector is fitted on all rows, so it sees the validation labels.
selector = SelectKBest(f_classif, k=5).fit(X_demo, y_demo)
X_leaky = selector.transform(X_demo)
leaky_scores = cross_val_score(
    LogisticRegression(max_iter=1000), X_leaky, y_demo, cv=demo_folds
)

# RIGHT: the selector is fitted inside each training fold.
honest_pipeline = Pipeline(
    [
        ("select", SelectKBest(f_classif, k=5)),
        ("clf", LogisticRegression(max_iter=1000)),
    ]
)
honest_scores = cross_val_score(
    honest_pipeline, X_demo, y_demo, cv=demo_folds
)

print(f"leaky  accuracy: {leaky_scores.mean():.3f}")
print(f"honest accuracy: {honest_scores.mean():.3f}")
print(f"gap:             {leaky_scores.mean() - honest_scores.mean():+.3f}")


leaky  accuracy: 0.787
honest accuracy: 0.575
gap:             +0.213


The gap is not a better model. It is information from the validation labels
that leaked into feature selection.

In our project the same thing happens if `SelectFromModel`, `StandardScaler`,
or an imputer is fitted before `cross_validate`. The fix is structural: put
every preprocessing step inside a `Pipeline` and let the CV loop fit the whole
thing on each training fold.


## 3. `ColumnTransformer`: one switchboard for column types

Different columns need different treatment. `ColumnTransformer` sends each
group to its own transformer and concatenates the results.

`set_output(transform="pandas")` keeps the output as a named `DataFrame`, which
we need later to map feature importances back to real columns.


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

people = pd.DataFrame(
    {
        "age": [25, 40, np.nan, 60],
        "city": ["Braga", "Porto", "Braga", None],
        "health": ["Bom", "Razoável", "Mau", "Bom"],
    }
)

preprocess = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median"), ["age"]),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["city"],
        ),
        ("ord", OrdinalEncoder(categories=[["Mau", "Razoável", "Bom"]]), ["health"]),
    ]
).set_output(transform="pandas")

preprocess.fit(people)
preprocess.transform(people)


,num__age,cat__city_Braga,cat__city_Porto,cat__city_nan,ord__health
0,25.0,1.0,0.0,0.0,2.0
1,40.0,0.0,1.0,0.0,1.0
2,40.0,1.0,0.0,0.0,0.0
3,60.0,0.0,0.0,1.0,2.0


## 4. `Pipeline`: keep `fit` inside the fold

A `Pipeline` chains the preprocessor and the classifier. `fit` on a training
fold fits the imputer, scaler, and encoder on that fold only. `predict` reuses
those fitted statistics on the validation fold.

`cross_validate(pipeline, X_raw, y, cv=folds)` clones the *whole pipeline* for
each fold. Nothing from the validation fold is ever seen by the preprocessor.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

y_people = pd.Series([0, 1, 2, 2])
people_pipeline = Pipeline(
    [
        ("prep", preprocess),
        ("clf", DecisionTreeClassifier(max_depth=2, random_state=42)),
    ]
)
people_pipeline.fit(people, y_people)
print("predictions:", people_pipeline.predict(people))
print("probabilities:")
print(people_pipeline.predict_proba(people).round(2))


predictions: [0 1 2 2]
probabilities:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]]


## 5. The project's shared machinery

`build_preprocessor` is just a `ColumnTransformer` built from our `ColumnSpec`.
`make_model` returns an unfitted classifier from the registry. `run_cv` does the
fold loop, computes the frozen metrics, and keeps out-of-fold predictions.

A notebook experiment is therefore: one `Pipeline` plus one `run_cv` call.


In [7]:
from daa.data import infer_column_spec
from daa.evaluation.cv import run_cv
from daa.features.preprocess import build_preprocessor
from daa.fixtures import make_synthetic_frame
from daa.models.registry import make_model
from daa.splits import SplitConfig, make_folds

frame = make_synthetic_frame(n_samples=240, seed=0)
X = frame.drop(columns=["AGE_RANGE"])
y = frame["AGE_RANGE"]

spec = infer_column_spec(X)
spec.counts()


{'numeric': 5, 'ordinal': 0, 'nominal': 3, 'drop': 2}

In [8]:
pipeline = Pipeline(
    [
        ("prep", build_preprocessor(spec, variant="impute")),
        ("clf", make_model("decision_tree", max_depth=4, min_samples_leaf=5)),
    ]
)
folds = make_folds(y, SplitConfig(n_splits=3, n_repeats=1, seed=0))
result = run_cv(pipeline, X, y, folds, name="primer_demo")
result.metrics


{'macro_f1_mean': 0.4995620174047577,
 'macro_f1_std': 0.034716709580834386,
 'accuracy_mean': 0.5166666666666667,
 'accuracy_std': 0.023570226039551608,
 'qwk_mean': 0.7329152906665977,
 'qwk_std': 0.025796506078331203,
 'mae_bands_mean': 0.55,
 'mae_bands_std': 0.0510310363079829,
 'fit_time_mean': 0.021915376999459113,
 'fit_time_std': 0.008897949603141463,
 'score_time_mean': 0.012157889333442048,
 'score_time_std': 0.006174221066983854,
 'n_features': 14.0}

In [9]:
result.per_class


,label,precision,recall,f1,support
0,0,0.594203,0.706897,0.645669,58.0
1,1,0.433962,0.348485,0.386555,66.0
2,2,0.392857,0.372881,0.382609,59.0
3,3,0.612903,0.666667,0.638655,57.0


In [10]:
result.confusion


pred,0,1,2,3
true,,,,
0,41,13,4,0
1,24,23,13,6
2,4,15,22,18
3,0,2,17,38


## 6. Rules to take away

1. **Fit preprocessing inside the pipeline.** If you transform before
   `cross_validate`, you are leaking.
2. **Never pass the test set to `fit`/`fit_transform`.** The test set is for the
   final submission only.
3. **Keep column roles in `ColumnSpec`.** Do not hard-code column lists in
   notebooks; that is how train and test drift apart.
4. **If a result is worth keeping, record it.** Use `run_experiment` or
   `run_cv` + `write_run` so it lands in `outputs/leaderboard.csv`.
5. **One seed, one set of folds.** `SEED` and `splits/folds.npz` are frozen.
